In [19]:
%load_ext autoreload
%autoreload 2

from meles.recognizer.arcface import ArcFaceRecognizer
from meles.aggregation.aggregator import NoAggregator
from meles.classifier import Classifier
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Allow the usage of progress bars on pandas dataframes
tqdm.pandas()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# Initialize and configure the recognizer, aggregator and classifier. The classifier has to use the metric of the recognizer.
aggregator = NoAggregator(ArcFaceRecognizer())
classifier = Classifier(metric=aggregator.metric())

In [3]:
# Load the dataset from the experiment structure file.
DATA_PATH = Path("../../data")

experiment = pd.read_json(DATA_PATH.joinpath("ytfaces_experiment.json"))
experiment

,identity,video,frame,frame_suffix,path
0,Aaron_Eckhart,0,aligned_detect_0.555.jpg,.jpg,ytfaces/Aaron_Eckhart/0/aligned_detect_0.555.jpg
1,Aaron_Eckhart,0,aligned_detect_0.556.jpg,.jpg,ytfaces/Aaron_Eckhart/0/aligned_detect_0.556.jpg
2,Aaron_Eckhart,0,aligned_detect_0.557.jpg,.jpg,ytfaces/Aaron_Eckhart/0/aligned_detect_0.557.jpg
3,Aaron_Eckhart,0,aligned_detect_0.558.jpg,.jpg,ytfaces/Aaron_Eckhart/0/aligned_detect_0.558.jpg
4,Aaron_Eckhart,0,aligned_detect_0.559.jpg,.jpg,ytfaces/Aaron_Eckhart/0/aligned_detect_0.559.jpg
...,...,...,...,...,...
130722,Zulfiqar_Ahmed,4,aligned_detect_4.225.jpg,.jpg,ytfaces/Zulfiqar_Ahmed/4/aligned_detect_4.225.jpg
130723,Zulfiqar_Ahmed,4,aligned_detect_4.226.jpg,.jpg,ytfaces/Zulfiqar_Ahmed/4/aligned_detect_4.226.jpg
130724,Zulfiqar_Ahmed,4,aligned_detect_4.227.jpg,.jpg,ytfaces/Zulfiqar_Ahmed/4/aligned_detect_4.227.jpg
130725,Zulfiqar_Ahmed,4,aligned_detect_4.228.jpg,.jpg,ytfaces/Zulfiqar_Ahmed/4/aligned_detect_4.228.jpg


In [4]:
# Keep the first three videos of every identity in the train set and move the rest into the test set.
TRAIN_VIDEOS_PER_IDENTITY = 3

# Rank the distinct videos of each identity by their natural (ascending) order, so
# that the three lowest-numbered videos of every identity receive ranks 1, 2 and 3.
video_rank = experiment.groupby("identity")["video"].transform(
    lambda videos: videos.rank(method="dense")
)
is_train = video_rank <= TRAIN_VIDEOS_PER_IDENTITY

train = experiment[is_train].reset_index(drop=True)
test = experiment[~is_train].reset_index(drop=True)

# Every identity keeps at least its first video in the train set, so all test identities
# are guaranteed to be present in the train set as well.
assert set(test["identity"]).issubset(set(train["identity"]))

print(f"Train: {len(train):>8,} frames | {train.groupby(['identity', 'video']).ngroups:>5,} videos | {train['identity'].nunique():>4,} identities")
print(f"Test:  {len(test):>8,} frames | {test.groupby(['identity', 'video']).ngroups:>5,} videos | {test['identity'].nunique():>4,} identities")

Train:  110,367 frames | 1,599 videos |  533 identities
Test:    20,360 frames |   293 videos |  226 identities


In [21]:
# For every identity, embed all frames into a list of embeddings (with potentially different size). Then, fit the
train_embeddings = (
    train.groupby('identity')['path']
    .progress_apply(lambda paths: aggregator.embed([str(DATA_PATH / path) for path in paths]))
    .explode()
    .rename('embedding')
    .reset_index()
    .dropna(subset=['embedding'])
)
# TODO incorporate the aggregator name into the output file
train_embeddings.to_json(DATA_PATH / "ytfaces_train.json")
train_embeddings

  0%|          | 1/533 [00:07<1:06:21,  7.48s/it]


KeyboardInterrupt: 